# DESC Supernovae (SN) Task Force metric - counting well-measured Type Ia SNe in MAF


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-10
- Context: SCOC (Survey Cadence Optimization Committee) - DESC Task Force metrics, restricted to the DESC metrics only (3x2pt, Weak Lensing, Supernovae)
- This notebook: **SN** (expected number of well-measured Type Ia supernovae, `SNNSNMetric`)
- Companion notebooks in this series (`06_MAF_DESC_TaskF`): `01_3x2pts_DESC_TaskForce_demo.ipynb` (3x2pt static-probes FoM) and `02_WL_DESC_TaskForce_demo.ipynb` (weak-lensing systematics-mitigation proxies).
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Reproduces the official `rubin_sim.maf.batches.science_radar_batch` "Cosmology" / "5: SNe Ia" subgroup, so results are directly comparable to the standard show_maf pages for this OpSim run.


## Notebook overview

**What "counting supernovae" means for DESC/SCOC.** Not every SNIa that explodes in the survey footprint is useful for cosmology: a light curve only yields a good standardized distance if it is well sampled around maximum light, in enough bands, with high enough signal-to-noise. `rubin_sim.maf` implements this selection with a single metric, **`SNNSNMetric`**, which for every Healpix pixel and observing season:

1. simulates fast SALT2-like Type Ia light curves (via a pre-computed grid, `LcfastNew`) using the actual cadence, filter sequence and per-visit 5-sigma depths observed at that pixel, converted into realistic photometric uncertainties with a pre-tabulated `gamma` reference table (`gamma_WFD.hdf5`);
2. defines a light curve as *well measured* if it has at least `n_bef` points before peak and `n_aft` points after peak, each above a minimum SNR - the same practical criterion used in DESC's LSST Deep Drilling / Wide-Fast-Deep SN forecasts (Gris et al. 2023, ApJS 264, 22);
3. finds the **redshift completeness limit `zlim`**: the redshift up to which a fraction `zlim_coeff` (default 95%) of simulated SNeIa (for canonical stretch/color x1=0, c=0) are well measured, using a realistic volumetric SN rate model (`SnRate`);
4. **counts `n_sn`**: the expected number of well-measured SNeIa up to that `zlim`, again from the volumetric rate model, integrated over the pixel area and the season length.

Because the metric's `run()` method returns two numbers per pixel (`n_sn`, `zlim`) packed into a single structured value, MAF's automatic **reduce-function** mechanism splits them into two independent Healpix maps as soon as the bundle group is run: `SNNSNMetric_n_sn` and `SNNSNMetric_zlim`. Summing the `SNNSNMetric_n_sn` map over the whole sky (a `SumMetric` summary statistic, labeled `"Total detected"` in the official batch) gives the headline number: **the total number of well-measured Type Ia supernovae the cadence is expected to deliver** - exactly the count requested for this notebook.

This notebook reproduces the official batch configuration for the Wide-Fast-Deep (WFD) survey: `nside=16` (much coarser than the 64 used for 3x2pt/WL, because this metric is expensive per pixel), Deep-Drilling-Field visits excluded, redshift range 0.2-0.5, `n_bef=3` / `n_aft=8`.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/metrics/sn_n_sn_metric.py` (`SNNSNMetric`, including the `reducen_sn` / `reducezlim` reduce functions)
  - `rubin_sim/maf/batches/science_radar_batch.py` (official "Cosmology" batch definition, "SNe Ia" subgroup, function `science_radar_batch`)
- summary.h5 / MAF outputs for standard runs: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Table of simulations: https://usdf-maf.slac.stanford.edu/


## Practical note before running

`SNNSNMetric` is **much more expensive per pixel** than the depth/visit-counting metrics used in notebooks 01 and 02: it simulates light curves and interpolates a reference grid at every Healpix pixel and season. This is why the official batch drops `nside` from 64 down to **16** for this metric. On a laptop, a full 10-year, non-DD, `nside=16` run can still take several minutes. It also needs the SN template / `gamma_WFD.hdf5` reference files that ship with the `rubin_sim` data package - if this raises a file-not-found error, run `rs_download_data --dirs maf` (or the relevant `rs_download_data` target for SN templates) to populate `$RUBIN_SIM_DATA_DIR` first.

## 1. Imports

In [ ]:
import os
import time
import inspect
from os.path import splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.metric_bundles as mb

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

Same OpSim file and output-directory convention as notebooks 01/02, with a dedicated `NB_TAG`.

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
NB_TAG = "SN"
data_dir = f"data_03_{NB_TAG}"
figs_dir = f"figs_03_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF metric class used to count DESC-usable SNeIa

In [ ]:
from rubin_sim.maf.metrics.sn_n_sn_metric import SNNSNMetric

print(inspect.getdoc(SNNSNMetric))

In [ ]:
# The two summary quantities the metric produces per pixel, via its reduce functions:
print("reduce_funcs:", list(SNNSNMetric().reduce_funcs.keys()))
# %psource SNNSNMetric.run       # uncomment to inspect the full per-pixel light-curve selection logic
# %psource SNNSNMetric.reducen_sn
# %psource SNNSNMetric.reducezlim

## 4. Configuration (matching the official `science_radar_batch` "SNe Ia" subgroup)

These are exactly the parameters used to populate the official MAF "Cosmology / 5: SNe Ia" show_maf pages: a coarse `nside=16` slicer (for tractable run time), the Wide-Fast-Deep footprint only (Deep-Drilling-Field visits excluded via `scheduler_note not like '%DD%'`), a shallow low-z selection (0.2 <= z <= 0.5) with `n_bef=3` points before peak and `n_aft=8` after, and a 95% redshift-completeness target.

In [ ]:
sne_nside = 16
sqlconstraint = "scheduler_note not like '%DD%'"
info_label = "DDF excluded"

sn_metric = metrics.SNNSNMetric(
    n_bef=3,
    n_aft=8,
    coadd_night=True,
    add_dust=False,
    hard_dust_cut=0.25,
    zmin=0.2,
    zmax=0.5,
    z_step=0.03,
    daymax_step=3.0,
    zlim_coeff=0.95,
    gamma_name="gamma_WFD.hdf5",
    verbose=False,
)

sn_summary = [
    metrics.MedianMetric(),
    metrics.MeanMetric(),
    metrics.SumMetric(metric_name="Total detected"),
    metrics.CountMetric(metric_name="Total on sky", mask_val=0),
]

snslicer = slicers.HealpixSlicer(nside=sne_nside, use_cache=False)
pix_area_sn = hp.nside2pixarea(sne_nside, degrees=True)
print(
    f"sne_nside={sne_nside} -> pixel area = {pix_area_sn:.2f} deg^2 "
    f"({hp.nside2npix(sne_nside)} pixels over the full sky)"
)

## 5. Running the metric

A single bundle, over the full 10-year, non-DD footprint (no year-by-year loop here, unlike notebooks 01 and 02 - the official batch itself only evaluates this metric once, for the full survey, given its cost). MAF automatically splits the complex (`n_sn`, `zlim`) output into two Healpix-map bundles via the metric's reduce functions, named `SNNSNMetric_n_sn` and `SNNSNMetric_zlim`.

In [ ]:
plot_dict = {"percentile_clip": 95, "n_ticks": 5}

sn_bundle = mb.MetricBundle(
    sn_metric,
    snslicer,
    sqlconstraint,
    plot_dict=plot_dict,
    run_name=run_name,
    info_label=info_label,
    summary_metrics=sn_summary,
)

bd = mb.make_bundles_dict_from_list([sn_bundle])
bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=data_dir, results_db=resultsDb)

t0 = time.time()
bgroup.run_all()
print(f"Done in {time.time() - t0:.1f} s")

In [ ]:
# The reduce functions added two new bundles to bgroup.bundle_dict:
print(sorted(bgroup.bundle_dict.keys()))

nsn_bundle = bgroup.bundle_dict["SNNSNMetric_n_sn"]
zlim_bundle = bgroup.bundle_dict["SNNSNMetric_zlim"]

## 6. Headline result: total number of well-measured SNeIa

In [ ]:
summary_table = pd.DataFrame(
    {
        "n_sn (per pixel)": nsn_bundle.summary_values,
        "zlim (per pixel)": zlim_bundle.summary_values,
    }
).rename_axis("summary statistic")
summary_table

In [ ]:
total_sn = nsn_bundle.summary_values["Total detected"]
median_zlim = zlim_bundle.summary_values["Median"]
print(
    f"Total number of well-measured Type Ia SNe expected ({info_label}, "
    f"{sn_metric.zmin} <= z <= {sn_metric.zmax}): {total_sn:.0f}"
)
print(f"Median redshift completeness limit (zlim) across the footprint: {median_zlim:.3f}")

In [ ]:
summary_csv = os.path.join(data_dir, f"{run_name}_SN_summary.csv")
summary_table.to_csv(summary_csv)
print("Saved:", summary_csv)

## 7. Healpix maps and histograms

As in notebooks 01 and 02, we use the Healpix slicer's own default `HealpixSkyMap` + `HealpixHistogram` plotters (`bundle.plot()`) - the native MAF rendering for both the per-pixel SN counts and the per-pixel redshift completeness limit.

In [ ]:
def save_bundle_plots(bundle, tag, figs_dir):
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved

In [ ]:
saved = save_bundle_plots(nsn_bundle, f"{run_name}_SNNSNMetric_n_sn", figs_dir)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = nsn_bundle.plot(savefig=False)
plt.show()

In [ ]:
saved = save_bundle_plots(zlim_bundle, f"{run_name}_SNNSNMetric_zlim", figs_dir)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = zlim_bundle.plot(savefig=False)
plt.show()

## 8. n_sn vs zlim: is the count driven by area, depth, or both?

In [ ]:
nsn_vals = nsn_bundle.metric_values.filled(np.nan)
zlim_vals = zlim_bundle.metric_values.filled(np.nan)
good = np.isfinite(nsn_vals) & np.isfinite(zlim_vals) & (nsn_vals > 0)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(zlim_vals[good], nsn_vals[good], s=4, alpha=0.3, c="teal")
ax.set_xlabel("Redshift completeness limit, zlim (per pixel)")
ax.set_ylabel("Number of well-measured SNeIa, n_sn (per pixel)")
ax.set_title(f"DESC SN Task Force metric - {run_name}")
ax.grid(alpha=0.3)
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_SN_nsn_vs_zlim")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 9. Caveats

- The `zmin=0.2, zmax=0.5` window and `n_bef=3` / `n_aft=8` selection reproduce the *official* WFD entry of `science_radar_batch`. This is a relatively shallow, low-redshift, WFD-optimized SN sample - it is *not* the deep, DDF-based SN cosmology sample (which uses different criteria, typically `n_bef=4` / `n_aft=10`, over a much smaller but much more intensively observed footprint; see Gris et al. 2023 for the DDF design study). This notebook, matching the rest of this `06_MAF_DESC_TaskF` series, restricts to the non-DD (Wide-Fast-Deep) footprint only.
- `n_sn` and `zlim` are computed independently per season, then `n_sn` is summed and `zlim` is the median over seasons with `zcomp > 0` (see `SNNSNMetric.run`); pixels/seasons with fewer than `n_bef + n_aft` usable observations, or with `E(B-V)` above `hard_dust_cut`, are masked out entirely (`badval`) rather than contributing a zero.
- Because `nside=16` is coarse (pixel area ~13.4 deg^2), the sky maps in Section 7 are necessarily blocky; this is a deliberate trade-off for run time, matching the official batch, not a bug.
- `SNNSNMetric` relies on pre-computed fast light-curve and photometric-uncertainty (`gamma`) reference grids shipped with the `rubin_sim` data package; if a different `gamma_name` or redshift range is needed, check that the corresponding reference file is available before changing these parameters.


## References
- Gris, Ph. et al. 2023, "Designing an Optimal LSST Deep Drilling Program for Cosmology with Type Ia Supernovae", ApJS 264, 22 - defines the well-measured-light-curve selection criteria implemented by `SNNSNMetric`.
- Bianco, F. B. et al. 2022, "Optimization of the Observing Cadence for the Rubin Observatory LSST: A Pioneering Process of Community-Focused Experimental Design", ApJS 258, 1 - SCOC context.
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/sn_n_sn_metric.py`, `rubin_sim/maf/batches/science_radar_batch.py`)
